In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
use catalog lendingclub;
create schema if not exists silver_cleaned;
use schema silver_cleaned;
select current_catalog(), current_schema();

In [0]:
loans_repayment_df = spark.read.table('bronze.LoanRepayment')

In [0]:
#ingestion date

loans_p_ingested_df = loans_repayment_df.withColumn('ingested_Date', current_timestamp())

In [0]:
#check if payment related column is null and remove those values

loans_p_ingested_df.filter('last_payment_amount is null').count()

In [0]:
columns_to_consider = ['total_rec_principal','total_rec_interest','total_rec_late_fee','total_payment', 'last_payment_amount']

loans_p_modified_df = loans_p_ingested_df.dropna(subset = columns_to_consider)

display(loans_p_modified_df.head(5))

In [0]:
loans_p_ingested_df.count()

In [0]:
loans_p_modified_df.count()

In [0]:
#check if we have total_payment is 0 but total_payment_received is not 0
#if we have any such entries then its wrongly calculating

display(loans_p_modified_df.filter("total_payment == '0.0' and total_rec_principal != '0.0'"))

In [0]:
#we need to calculate total_payment in this cases
# total_payment = total_rec_principal + total_rec_interest + total_rec_late_fee

loans_p_calc_df = loans_p_modified_df.withColumn(
    'total_payment',
    when(
        (col('total_payment') == 0.0) & (col('total_rec_principal') != 0.0 ),
        col('total_rec_principal')+ col('total_rec_interest')+ col('total_rec_late_fee')
    ).otherwise(col('total_payment'))
)

In [0]:
#check if we have 0 total payments

loans_p_calc_df.filter("total_payment == '0.0'").count()

In [0]:
loans_p_fixed_df = loans_p_calc_df.filter("total_payment != '0.0'")

In [0]:
#check the last_payment_Date and next_payment_date with 0.0

display(loans_p_fixed_df.filter("last_payment_date == '0.0'"))

In [0]:
loans_p_fixed_df.filter("last_payment_date == '0.0'").count()

In [0]:
loans_p_fixed_df.filter("next_payment_date == '0.0'").count()

In [0]:
#hence we are having some goo data for this values- we can replace 0.0 with null

loans_p_ldatefixed_df = loans_p_fixed_df.withColumn(
    'last_payment_date',
    when(col('last_payment_date') == '0.0',
         None
        )
        .otherwise(col('last_payment_date'))
)

In [0]:
loans_p_ndatefixed_df = loans_p_ldatefixed_df.withColumn(
    'next_payment_date',
    when(col('next_payment_date') == '0.0',
         None
        )
        .otherwise(col('next_payment_date'))
)

In [0]:
display(loans_p_ndatefixed_df.head(5))

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/cleaned/LoansRepayment/", recurse=True)

In [0]:
loans_p_ndatefixed_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/cleaned/LoansRepayment/')

In [0]:
%sql
create or replace table silver_cleaned.loans_repayment
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/cleaned/LoansRepayment/`